In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import output
import threading
import time
from tetris import TetrisGame

# --- ゲーム設定 ---
GAME_SPEED = 0.4  # ブロックが1段落ちるまでの秒数（0.5から0.4に調整）
BLOCK_SYMBOLS = {
    0: '⬜',
    1: '🟦',
    2: '🟨',
    3: '🟪',
    4: '🟧',
    5: '🟦',
    6: '🟩',
    7: '🟥',
}
BOARD_WIDTH = 10
BOARD_HEIGHT = 20

# --- グローバル変数 ---
game = TetrisGame(BOARD_WIDTH, BOARD_HEIGHT)
game_thread = None
stop_game_flag = False

# --- UIウィジェットの作成 ---
game_output = widgets.Output()
score_label = widgets.Label(value=f"Score: {game.score}")
game_over_label = widgets.Label(value="")
start_button = widgets.Button(description='Start Game', button_style='success')

# ボタンコントロールを削除し、キーボード操作の案内を追加
controls_info = widgets.Label(value="操作: ← → (移動), ↑ (ハードドロップ), ↓ (ソフトドロップ), Space (回転)")
ui = widgets.VBox([score_label, game_over_label, game_output, controls_info, start_button])

# --- 描画関数 ---
def draw_game_state():
    with game_output:
        clear_output(wait=True)
        board_state = game.get_board_state()
        board_html = "<pre style='font-family: monospace; line-height: 1;'>"
        for row in board_state:
            board_html += ''.join([BLOCK_SYMBOLS.get(cell, ' ') for cell in row]) + '\n'
        board_html += "</pre>"
        display(HTML(board_html))
        score_label.value = f"Score: {game.score}"
        if game.game_over:
            game_over_label.value = "GAME OVER"

# --- ゲームループ ---
def game_loop():
    global stop_game_flag
    while not game.game_over and not stop_game_flag:
        game.step()
        draw_game_state()
        time.sleep(GAME_SPEED)
    
    if game.game_over:
        start_button.description = 'Restart'
        start_button.disabled = False
        start_button.button_style = 'warning'

# --- キーボードイベント処理 ---
def handle_key_event(key):
    if not game.game_over:
        if key == 'ArrowLeft':
            game.move(-1)
        elif key == 'ArrowRight':
            game.move(1)
        elif key == 'ArrowDown':
            game.step() # ソフトドロップ
        elif key == 'ArrowUp':
            game.drop() # ハードドロップ
        elif key == ' ':
            game.rotate()
        draw_game_state()

output.register_callback('notebook.handle_key_event', handle_key_event)

# --- スタートボタンのコールバック関数 ---
def on_start_button_clicked(b):
    global game, game_thread, stop_game_flag
    
    if game_thread and game_thread.is_alive():
        stop_game_flag = True
        game_thread.join()
        
    game = TetrisGame(BOARD_WIDTH, BOARD_HEIGHT)
    stop_game_flag = False
    score_label.value = f"Score: {game.score}"
    game_over_label.value = ""
    start_button.disabled = True
    
    game_thread = threading.Thread(target=game_loop)
    game_thread.start()
    draw_game_state()

# --- イベントハンドラを登録 ---
start_button.on_click(on_start_button_clicked)

# --- JavaScriptでキー入力をキャプチャしてPythonに送る ---
js_code = """
<script>
document.addEventListener('keydown', function(e) {
  // ゲームで使うキーの場合はページのスクロールなどを防ぐ
  if ([' ', 'ArrowUp', 'ArrowDown', 'ArrowLeft', 'ArrowRight'].indexOf(e.key) > -1) {
    e.preventDefault();
  }
  // Python側のコールバック関数を呼び出す
  google.colab.kernel.invokeFunction('notebook.handle_key_event', [e.key], {});
});
</script>
"""

# --- アプリケーションの表示 ---
display(ui)
display(HTML(js_code)) # JSを注入
draw_game_state() # 初期盤面を描画